# **FIRST MODEL**

# --- CELL 1: setup -------------------------------------------

In [ ]:
!pip -q install lightgbm pyarrow
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, subprocess, importlib

ZIP = '/content/drive/MyDrive/riverload-data.zip'
assert os.path.exists(ZIP), f'zip not found at {ZIP}'

hits = [h for h in glob.glob('/content/**/riverload/__init__.py', recursive=True) if '__MACOSX' not in h]
if not hits:
    subprocess.run(f'unzip -q -o "{ZIP}" -d /content', shell=True, check=True)
    hits = [h for h in glob.glob('/content/**/riverload/__init__.py', recursive=True) if '__MACOSX' not in h]
assert hits, 'riverload package not found after unzip'

DATA = os.path.dirname(os.path.dirname(hits[0]))
print('DATA =', DATA)
print('package files:', sorted(os.listdir(f'{DATA}/riverload')))
print('__init__ size:', os.path.getsize(f'{DATA}/riverload/__init__.py'), 'bytes')

# make sure nothing else called "riverload" shadows it, and clear every import cache
for m in [k for k in sys.modules if k.startswith('riverload') or k == 'to_kaggle']:
    del sys.modules[m]
sys.path = [p for p in sys.path if p not in (DATA, '', '/content')]
sys.path.insert(0, DATA)
sys.path_importer_cache.clear()
importlib.invalidate_caches()

import riverload
print('riverload loaded from:', riverload.__file__)     # must be .../riverload-data/riverload/__init__.py

PLAN_Q, BUFFER_CM = 'c0.1', 10.0
RUN_OOF = True
STORED_OFFSETS = {0.05: -10.8, 0.1: -8.9, 0.2: -6.9, 0.5: -3.6}

import time, numpy as np, pandas as pd, lightgbm as lgb
from riverload import Env
from riverload.data import Data
from riverload.quickstart import decide
from to_kaggle import c2_to_kaggle

QS = [0.05, 0.10, 0.20, 0.50]
ROUTES = ['A1', 'A2', 'B1', 'B2']
print('imports OK')

# CELL 2: features

In [ ]:
env_tr = Env(DATA, 'train'); data_tr = Data(DATA, 'train', env=env_tr)
seg = data_tr.segments[data_tr.segments.segment_type == 'RIVER'].set_index('segment_id')

def template_tt():
    out = []
    for fam in ('A', 'B'):
        sc = env_tr.scenarios[env_tr.scenarios.family == fam].iloc[0]
        tt = env_tr.timetable(sc.scenario_id); tt = tt[tt.kind == 'RIVER']
        out.append(tt.assign(a=np.floor((tt.t_enter - sc.issue_time) / pd.Timedelta(hours=1)).astype(int),
                             b=np.floor((tt.t_exit - sc.issue_time) / pd.Timedelta(hours=1)).astype(int)))
    return pd.concat(out)
TT = template_tt()
LS = {k: sorted(g.segment_id.unique()) for k, g in TT.groupby(['route_id', 'leg'])}

def depth_df(lev):
    d = pd.DataFrame({s: seg.loc[s, 'guaranteed_depth_cm'] + (lev[seg.loc[s, 'gauge_id']] - seg.loc[s, 'ref_level_shifted_cm'])
                      for s in seg.index})
    return d.ffill().bfill()

def make_rows(issue_idx, dep, lev, q, with_target=True):
    D = dep.to_numpy(float); cols = {c: i for i, c in enumerate(dep.columns)}; I = np.asarray(issue_idx)
    legfeat = {}
    for k, segs in LS.items():
        s = pd.Series(np.min(D[:, [cols[x] for x in segs]], axis=1)); r21 = s.rolling(504, min_periods=24)
        legfeat[k] = dict(now=s.shift(1), d1=s.shift(1) - s.shift(25), d3=s.shift(1) - s.shift(73), d7=s.shift(1) - s.shift(169),
                          mn=r21.min().shift(1), mx=r21.max().shift(1), mean=r21.mean().shift(1),
                          mn3=s.rolling(72, min_periods=12).min().shift(1))
    Lv = lev.reset_index(drop=True); Qv = q.reset_index(drop=True)
    glob_ = pd.concat([Lv.shift(1).add_prefix('lv_'), (Lv.shift(1) - Lv.shift(25)).add_prefix('dl1_'),
                       (Lv.shift(1) - Lv.shift(73)).add_prefix('dl3_'), Qv.shift(1).add_prefix('q_'),
                       (Qv.shift(1) - Qv.shift(73)).add_prefix('dq3_')], axis=1).iloc[I].reset_index(drop=True)
    doy = dep.index[I].dayofyear.to_numpy()
    glob_['doy_s'] = np.sin(2 * np.pi * doy / 365.25); glob_['doy_c'] = np.cos(2 * np.pi * doy / 365.25)
    out, rmq = [], {}
    for (rid, slot, leg), g in TT.groupby(['route_id', 'departure_slot', 'leg']):
        base = pd.DataFrame({k: v.to_numpy()[I] for k, v in legfeat[(rid, leg)].items()})
        base['route'] = ROUTES.index(rid); base['lek'] = int(rid[1] == '2'); base['fam'] = int(rid[0] == 'B')
        base['slot'] = slot; base['leg'] = leg; base['lead_a'] = g.a.min(); base['lead_b'] = g.b.max()
        base['route_id'] = rid; base['issue_pos'] = I
        if with_target:
            m = np.full(len(I), np.inf)
            for _, row in g.iterrows():
                w = row.b - row.a + 1; key = (row.segment_id, w)
                if key not in rmq:
                    rmq[key] = pd.Series(D[:, cols[row.segment_id]]).rolling(w).min().shift(-(w - 1)).to_numpy()
                idx = I + row.a; v = np.full(len(I), np.nan); ok = idx < len(D); v[ok] = rmq[key][idx[ok]]
                m = np.where(np.isnan(v) | np.isnan(m), np.nan, np.minimum(m, v))
            base['dmin'] = m
        out.append(pd.concat([base, glob_], axis=1))
    X = pd.concat(out, ignore_index=True)
    X['time'] = dep.index[X.issue_pos.to_numpy()]; X['year'] = pd.DatetimeIndex(X['time']).year
    return X

EXCL = {'route_id', 'issue_pos', 'dmin', 'time', 'year', 'y'}
feat_cols = lambda X: [c for c in X.columns if c not in EXCL]

# CELL 3: training data from history

In [ ]:
lev, qd = data_tr._lev.copy(), data_tr._q.copy()
dep = depth_df(lev)
days = pd.date_range('2015-02-15 06:00', '2021-12-10 06:00', freq='2D')
S = make_rows(np.array([dep.index.get_loc(t) for t in days]), dep, lev, qd)
S = S[S.slot % 3 == 0].dropna(subset=['dmin', 'now']).reset_index(drop=True)
S['y'] = S.dmin - S.now
F = feat_cols(S); print(S.shape, len(F), 'features')
P = dict(objective='quantile', learning_rate=0.05, num_leaves=31, min_data_in_leaf=200, feature_fraction=0.6,
         bagging_fraction=0.7, bagging_freq=1, lambda_l2=10, verbose=-1, num_threads=-1)
fit = lambda tr, qq: lgb.train({**P, 'alpha': qq}, lgb.Dataset(tr[F], tr.y), num_boost_round=150)

# CELL 4: leave-one-year-out forecasts and calibration

In [ ]:
def real_rows(env, dep_, lev_, q_):
    pos = np.array([dep_.index.get_loc(t) for t in env.scenarios.issue_time.unique()])
    return make_rows(pos, dep_, lev_, q_, with_target=False)

def attach(R, env, cols):
    sc = env.scenarios[['scenario_id', 'issue_time', 'family']].rename(columns={'issue_time': 'time'})
    sc['fam'] = (sc.family == 'B').astype(int)
    m = R.merge(sc, on=['time', 'fam']).rename(columns={'slot': 'departure_slot'})
    return m[['scenario_id', 'route_id', 'departure_slot', 'leg', 'now'] + cols]

offsets = STORED_OFFSETS
if RUN_OOF:
    Rr = real_rows(env_tr, dep, lev, qd)
    for qq in QS: Rr[f'p{qq}'] = np.nan
    for y in range(2015, 2022):
        tr = S[(S.time < pd.Timestamp(f'{y}-01-01') - pd.Timedelta(days=14)) | (S.time >= pd.Timestamp(f'{y+1}-01-01') + pd.Timedelta(days=14))]
        te = Rr.year == y
        for qq in QS:
            Rr.loc[te, f'p{qq}'] = Rr.loc[te, 'now'] + fit(tr, qq).predict(Rr.loc[te, F])
        print(y, 'done', flush=True)
    oof = attach(Rr, env_tr, [f'p{q}' for q in QS])
    mm = oof.merge(env_tr.truth_dmin, on=['scenario_id', 'route_id', 'departure_slot', 'leg'])
    offsets = {qq: float(np.quantile(mm.dmin_cm - mm[f'p{qq}'], qq)) for qq in QS}
    print('calibration offsets', offsets)
    for qq in QS: oof[f'c{qq}'] = oof[f'p{qq}'] + offsets[qq]

# CELL 5: validate the decision on train

In [ ]:
def validate(plan_q=PLAN_Q, buf=BUFFER_CM, every=5, slots=None):
    ids = env_tr.scenarios.scenario_id.iloc[::every]; sub = env_tr.subset(ids)
    f = oof[oof.scenario_id.isin(set(ids))]
    dec, lp = decide(sub, f, buffer_cm=buf, slots=slots, plan_col=plan_q)
    r = sub.evaluate(dec, lp)['c2']; return round(r['score'], 4), r['n_failed']

if RUN_OOF:
    print('(score, groundings):', validate())

# CELL 6: final model, test predictions and submission.csv

In [ ]:
env_te = Env(DATA, 'test'); te_sc = env_te.scenarios
Wl = pd.read_parquet(f'{DATA}/test/window_levels_hourly.parquet'); Wq = pd.read_parquet(f'{DATA}/test/window_discharge_hourly.parquet')
final = {qq: fit(S, qq) for qq in QS}

parts = []
for iid, sc_i in te_sc.groupby('issue_id'):
    it = sc_i.issue_time.iloc[0]
    l = Wl[Wl.issue_id == iid].set_index('timestamp').drop(columns='issue_id').sort_index()
    q_ = Wq[Wq.issue_id == iid].set_index('timestamp').drop(columns='issue_id').sort_index()
    l = l.reindex(pd.date_range(it - pd.Timedelta(hours=504), it, freq='h')).ffill().bfill()
    q_ = q_.reindex(l.index).ffill().bfill()
    d_ = depth_df(l)
    R = make_rows([len(l) - 1], d_, l, q_, with_target=False); R['time'] = it
    for qq in QS: R[f'c{qq}'] = R['now'] + final[qq].predict(R[F]) + offsets[qq]
    s2 = sc_i[['scenario_id', 'issue_time', 'family']].rename(columns={'issue_time': 'time'}); s2['fam'] = (s2.family == 'B').astype(int)
    m = R.merge(s2, on=['time', 'fam']).rename(columns={'slot': 'departure_slot'})
    parts.append(m[['scenario_id', 'route_id', 'departure_slot', 'leg'] + [f'c{q}' for q in QS]])
fc_te = pd.concat(parts, ignore_index=True)
for a, b in zip([f'c{q}' for q in QS[:-1]], [f'c{q}' for q in QS[1:]]): fc_te[b] = np.maximum(fc_te[b], fc_te[a])
print(fc_te.shape, fc_te.isna().sum().sum(), 'NaN')

dec, lp = decide(env_te, fc_te, buffer_cm=BUFFER_CM, plan_col=PLAN_Q)
sub = c2_to_kaggle(dec, lp, env_te.manifests)
sample = pd.read_csv(f'{DATA}/sample_submission.csv')
assert set(sub.id) == set(sample.id) and list(sub.columns) == list(sample.columns), 'format mismatch'
sub.to_csv('submission.csv', index=False)
print(sub.shape); print(dec.route_id.value_counts().to_dict(), dec.departure_slot.describe()[['min', 'mean', 'max']].to_dict())
from google.colab import files; files.download('submission.csv')